# tACS Bandit: Main Analyses

**Study:** Examining the effects of theta-tACS over left DLPFC on reward-based learning across the adult lifespan

**Design:** Within-subject crossover (active vs. sham theta-tACS, 6 Hz)

This notebook orchestrates all pre-registered and exploratory analyses by calling modular functions. Individual modules contain the implementation details.

---
## 0. Setup & Imports

In [ ]:
# Standard
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = 'plotly_white'
pio.renderers.default = 'notebook_connected'

# Project modules
from config import *
from data_loading import load_all_subjects, summarize_loading
from exclusions import apply_all_exclusions
from sample_descriptives import run_sample_descriptives
from blinding_analysis import run_blinding_analysis
from wsls import run_wsls_analysis
from reversal_analysis import run_reversal_analysis
from rescorla_wagner import run_rw_analysis
from ddm import run_ddm_analysis
from eeg_theta import run_theta_analysis
from cognitive_merge import run_cognitive_merge
from hypothesis_tests import run_hypothesis_tests
from order_effects import run_order_effects_analysis
from correlations import run_correlation_analysis

# =============================================================================
# Analysis Toggles
# =============================================================================
# Set these to False to load cached results from master_subject_data.csv
# instead of re-fitting from scratch. Useful when iterating on downstream
# analyses without waiting for model fitting each time.

REFIT_RW = True       # Set False to load R-W params from master CSV
REFIT_DDM = True      # Set False to load DDM params from master CSV
REFIT_EXTENDED = True  # Set False to skip extended R-W models
INCLUDE_EXPLORATORY = True  # Set True to load exploratory measures (TEI, AQ, etc.)

MASTER_CSV = DATA_DIR.parent / 'master_subject_data.csv'

print(f'Study: {len(SUBJECT_INFO)} participants in SUBJECT_INFO')
print(f'Data directory: {DATA_DIR}')
print(f'\nAnalysis toggles:')
print(f'  REFIT_RW = {REFIT_RW}')
print(f'  REFIT_DDM = {REFIT_DDM}')
print(f'  REFIT_EXTENDED = {REFIT_EXTENDED}')
print(f'  INCLUDE_EXPLORATORY = {INCLUDE_EXPLORATORY}')
if MASTER_CSV.exists():
    print(f'  Master CSV found: {MASTER_CSV}')
else:
    print(f'  Master CSV not found — will be created after cognitive merge')

---
## 1. Data Loading & Preprocessing

In [ ]:
# Load all behavioral data
data = load_all_subjects()
summarize_loading(data)

In [ ]:
# Apply exclusion criteria
exclusion_results = apply_all_exclusions(data)

# Extract filtered datasets
data_h1 = exclusion_results['data_h1']       # H1 analyses: sham only, behavioral exclusions
data_h2 = exclusion_results['data_h2']       # H2 analyses: active vs sham, all exclusions
data_clean = exclusion_results['data_clean'] # All clean trials
run_exclusions = exclusion_results['run_exclusions']
h2_eligible = exclusion_results['h2_eligible']

print(f'\nH1-eligible subjects: {data_h1["subject_id"].nunique()}')
print(f'H2-eligible subjects: {len(h2_eligible)}')

---
## 2. Sample Descriptives & Demographics

In [ ]:
# Demographics and cap size analysis
desc_results = run_sample_descriptives()

---
## 3. Blinding Integrity

In [ ]:
# Signal detection analysis of blinding
blinding_results = run_blinding_analysis(data_clean)

if blinding_results['blinding_intact']:
    print('\n✓ Blinding appears intact')
else:
    print('\n⚠ Blinding may be compromised — interpret with caution')

---
## 4. Win-Stay / Lose-Shift Analysis

In [ ]:
# WSLS analysis for H1 and H2
wsls_results = run_wsls_analysis(data_h1, data_h2, data)

---
## 5. Reversal Learning Analysis

In [ ]:
# Reversal-locked accuracy and trials to criterion
rev_results = run_reversal_analysis(data, data_clean, conditions=['sham', 'active'])

---
## 6. Rescorla-Wagner Model

In [ ]:
# Fit R-W model (MLE and MAP)
# Toggle REFIT_RW at top of notebook to control whether fitting runs from scratch

if REFIT_RW:
    rw_results = run_rw_analysis(
        data_clean, 
        h2_subjects=h2_eligible,
        run_recovery=False  # Set True to run parameter recovery (slow)
    )
else:
    # Load from master CSV cache
    print('Loading R-W parameters from master CSV (REFIT_RW = False)')
    rw_results = {'rw_mle': None, 'rw_map': None}
    
    if MASTER_CSV.exists():
        cached = pd.read_csv(MASTER_CSV)
        cached['subject_id'] = cached['subject_id'].astype(str)
        
        rw_rows = []
        for _, row in cached.iterrows():
            sid = row['subject_id']
            if pd.notna(row.get('sham_alpha')):
                rw_rows.append({'subject_id': sid, 'condition': 'sham',
                                'alpha': row['sham_alpha'], 'beta': row['sham_beta']})
            if pd.notna(row.get('active_alpha')):
                rw_rows.append({'subject_id': sid, 'condition': 'active',
                                'alpha': row['active_alpha'], 'beta': row['active_beta']})
        
        if rw_rows:
            rw_results['rw_mle'] = pd.DataFrame(rw_rows)
            print(f'  Loaded {len(rw_rows)} R-W parameter sets from cache')
        else:
            print('  WARNING: No R-W parameters found in master CSV — run with REFIT_RW = True')
    else:
        print('  WARNING: Master CSV not found — run with REFIT_RW = True first')

---
## 7. Drift Diffusion Model

In [ ]:
# Fit DDM (requires pyddm)
# Toggle REFIT_DDM at top of notebook to control whether fitting runs from scratch

if REFIT_DDM:
    ddm_results = run_ddm_analysis(
        data_clean,
        conditions=['sham', 'active'],
        run_bootstrap=False,  # Set True for bootstrap CI (slow)
        n_bootstrap=200
    )
else:
    # Load from master CSV cache
    print('Loading DDM parameters from master CSV (REFIT_DDM = False)')
    ddm_results = None
    
    if MASTER_CSV.exists():
        cached = pd.read_csv(MASTER_CSV)
        cached['subject_id'] = cached['subject_id'].astype(str)
        
        ddm_cols = [c for c in cached.columns if c.startswith('ddm_')]
        if ddm_cols:
            ddm_subject = cached[['subject_id'] + ddm_cols].dropna(subset=ddm_cols, how='all')
            ddm_results = {'ddm_subject': ddm_subject}
            print(f'  Loaded DDM parameters for {len(ddm_subject)} subjects ({len(ddm_cols)} params)')
        else:
            print('  WARNING: No DDM parameters found in master CSV — run with REFIT_DDM = True')
    else:
        print('  WARNING: Master CSV not found — run with REFIT_DDM = True first')

In [ ]:
print(type(ddm_results))
if isinstance(ddm_results, dict):
    print(ddm_results.keys())
    for k, v in ddm_results.items():
        if isinstance(v, pd.DataFrame):
            print(f'  {k}: {v.columns.tolist()[:10]}')

---
## 8. EEG Theta Reactivity

In [ ]:
# Compute theta reactivity from baseline EEG (F4 channel)
# Note: Requires EEG .easy files in DATA_DIR/nic/raw
try:
    theta_results = run_theta_analysis(verbose=True, show_plots=True)
    theta_subject = theta_results.get('theta_subject')
    print(f'\nTheta reliability (ICC): {theta_results["reliability"]["ICC"]:.3f}')
except FileNotFoundError:
    print('EEG files not found - skipping theta analysis')
    theta_results = None
    theta_subject = None

---
## 8b. Theta Reactivity Visualization

Visualize high vs. low theta reactivity exemplar subjects to illustrate individual differences, plus grand averages for win and loss feedback across all subs

In [ ]:
# Import theta visualization module
import importlib
import theta_visualization
importlib.reload(theta_visualization)
from theta_visualization import identify_theta_exemplars, preview_theta_timecourse, plot_theta_exemplar_comparison

In [ ]:
# Identify high and low theta reactivity subjects
exemplars = identify_theta_exemplars(
    theta_df=theta_results['theta_subject'],
    metric='theta_p95',
    verbose=True
)

In [ ]:
import importlib
import theta_visualization
importlib.reload(theta_visualization)
from theta_visualization import compute_grand_average_theta, plot_grand_average_theta

# Get list of subjects with earclip (cleaner data)
earclip_subjects = [s for s, info in SUBJECT_INFO.items() if info.get('earclip', True)]

print("Computing grand average feedback-locked theta...")
grand_avg = compute_grand_average_theta(
    subject_ids=earclip_subjects,
    run_nums=[1, 5],  # Baseline runs
    channel_idx=0,    # F4
    verbose=True
)

In [ ]:
if grand_avg:
    grand_avg_fig = plot_grand_average_theta(grand_avg, show_fig=True)

In [ ]:
from theta_visualization import plot_theta_timecourse_simple

simple_fig = plot_theta_timecourse_simple(
    high_subject_id='11318',
    low_subject_id='10608',
    run_num=1,
    channel_idx=0,
    timecourse_duration=60,
    show_fig=True
)

---
## 9. Cognitive & Survey Data Integration

In [ ]:
# Load external data and build subject DataFrame
cog_results = run_cognitive_merge(
    wsls_h1=wsls_results.get('wsls_h1'),
    wsls_h2=wsls_results.get('wsls_h2'),
    rw_mle=rw_results.get('rw_mle'),
    ddm_params=ddm_results.get('ddm_subject') if ddm_results else None,
    theta_subject=theta_subject,
    h2_subjects=h2_eligible,
    include_exploratory=INCLUDE_EXPLORATORY,
    export_csv=True,  # Write master_subject_data.csv
)

subj_df = cog_results['subj_df']
print(f'\nSubject DataFrame: {len(subj_df)} subjects, {len(subj_df.columns)} variables')
if 'theta_p95' in subj_df.columns:
    print(f'Subjects with theta data: {subj_df["theta_p95"].notna().sum()}')
if 'master_csv_path' in cog_results:
    print(f'Master CSV saved to: {cog_results["master_csv_path"]}')

---
## 9b. Extended Computational Models (Exploratory)

This section implements more sophisticated RL models to test hypotheses about **confirmation bias**, **perseveration**, and **value differentiation** in aging and tACS effects.

### Models
1. **Asymmetric R-W** (α⁺, α⁻, β): Separate learning rates for positive vs. negative prediction errors
2. **Asymmetric R-W + Stickiness** (α⁺, α⁻, β, τ): Adds perseveration independent of learned values
3. **Standard R-W + Stickiness** (α, β, τ): For model comparison

### Key Parameters
| Parameter | Interpretation |
|-----------|---------------|
| α⁺ | Learning rate for positive prediction errors (confirmatory feedback) |
| α⁻ | Learning rate for negative prediction errors (disconfirmatory feedback) |
| α⁺/α⁻ | Confirmation bias index (>1 = learns more from rewards than losses) |
| τ | Choice stickiness (>0 = perseveration, tendency to repeat regardless of value) |
| β | Inverse temperature / value differentiation |

### Hypotheses Tested
- **H-RL1**: Older adults show elevated α⁺/α⁻ ratios (confirmation bias)
- **H-RL2**: α⁺/α⁻ correlates with WSLS patterns (computational ↔ behavioral link)
- **H-RL5**: β declines with age (reduced value differentiation)
- **H-RL7**: Older adults show elevated τ (perseveration)
- **H-RL8**: τ correlates with poorer executive function

*Note: These analyses are exploratory extensions beyond the pre-registered hypotheses.*

In [ ]:
# Import extended R-W module
from rescorla_wagner_extended import run_extended_rw_analysis

In [ ]:
# Run extended R-W analysis
# Toggle REFIT_EXTENDED at top of notebook to skip this section

if REFIT_EXTENDED:
    extended_rw_results = run_extended_rw_analysis(
        data=data_clean,
        subj_df=subj_df,
        wsls_df=wsls_results['wsls_h1'],
        standard_rw_df=rw_results['rw_mle'],
        conditions=['sham', 'active'],
        method='mle',
        show_plots=True,
        verbose=True
    )
else:
    print('Skipping extended R-W analysis (REFIT_EXTENDED = False)')
    extended_rw_results = {
        'model_fits': pd.DataFrame(),
        'comparison': pd.DataFrame(),
        'hypothesis_tests': {},
        'sensitivity': {},
    }

In [ ]:
# Extract key results
extended_model_fits = extended_rw_results.get('model_fits', pd.DataFrame())
extended_comparison = extended_rw_results.get('comparison', pd.DataFrame())
extended_tests = extended_rw_results.get('hypothesis_tests', {})
sensitivity = extended_rw_results.get('sensitivity', {})

# Summary: Best model by subject
if len(extended_comparison) > 0 and 'best_model' in extended_comparison.columns:
    print("\nBest-fitting model distribution:")
    print(extended_comparison['best_model'].value_counts())
elif not REFIT_EXTENDED:
    print("Extended R-W was skipped (REFIT_EXTENDED = False)")

---
## 10. Pre-Registered Hypothesis Tests

In [ ]:
# Run all H1 and H2 tests
hyp_results = run_hypothesis_tests(subj_df, show_plots=True)

---
## 11. Order & Session Effects

In [ ]:
# Check for counterbalance confounds
order_results = run_order_effects_analysis(subj_df)

---
## 12. Correlation Matrix

In [ ]:
# Pairwise correlations among pre-registered variables
# (now includes theta_p95)
corr_results = run_correlation_analysis(subj_df, include_tertiary=False)

---
## 13. Summary

In [ ]:
print('='*70)
print('ANALYSIS SUMMARY')
print('='*70)
print()

# Sample
print(f'Sample: N = {len(subj_df)}')
print(f'  H1-eligible (sham): {data_h1["subject_id"].nunique()}')
print(f'  H2-eligible (paired): {len(h2_eligible)}')
if 'theta_p95' in subj_df.columns:
    print(f'  With theta data: {subj_df["theta_p95"].notna().sum()}')
print()

# Analysis toggles used
print(f'Analysis settings: REFIT_RW={REFIT_RW}, REFIT_DDM={REFIT_DDM}, REFIT_EXTENDED={REFIT_EXTENDED}')
print()

# Blinding
if blinding_results['blinding_intact']:
    print('✓ Blinding intact')
else:
    print('⚠ Blinding concern')

# Order effects
if order_results.get('any_significant_order', False):
    print('⚠ Significant order × condition interaction')
else:
    print('✓ No significant order effects')

# Theta reliability
if theta_results is not None and 'reliability' in theta_results:
    icc = theta_results['reliability'].get('ICC', np.nan)
    if not np.isnan(icc):
        if icc >= 0.75:
            print(f'✓ Theta reliability excellent (ICC = {icc:.2f})')
        elif icc >= 0.60:
            print(f'✓ Theta reliability good (ICC = {icc:.2f})')
        else:
            print(f'⚠ Theta reliability fair/poor (ICC = {icc:.2f})')

# Extended RL model comparison
if len(extended_comparison) > 0 and 'best_model' in extended_comparison.columns:
    best = extended_comparison['best_model'].value_counts().idxmax()
    pct = 100 * extended_comparison['best_model'].value_counts().max() / len(extended_comparison)
    print(f'✓ Best-fitting model: {best} ({pct:.0f}% of fits)')
print()

# H2.1 summary
print('H2.1 Paired Comparisons (tACS vs. Sham):')
for param, res in hyp_results['h2_1'].items():
    if res is not None:
        sig = '*' if res['p'] < 0.05 else ''
        print(f"  {param}: dz = {res['dz']:.3f}, p = {res['p']:.3f} {sig}")

# Master CSV location
if 'master_csv_path' in cog_results:
    print(f'\nMaster CSV: {cog_results["master_csv_path"]}')

---
## 14. Accuracy & Win Rate Analysis

Examines task performance metrics:
- **Run-level accuracy**: Performance across the 8 runs
- **Counterbalance effects**: Does order (sham-first vs active-first) matter?
- **Stimulation effects**: Does tACS affect accuracy or win rate?
- **Learning curves**: Within-run and within-contingency learning
- **Age effects**: Does accuracy decline with age?

In [ ]:
# Import accuracy analysis module
from accuracy_analysis import run_accuracy_analysis

In [ ]:
# Run complete accuracy analysis
acc_results = run_accuracy_analysis(
    data=data_clean,
    conditions=['sham', 'active'],
    show_plots=True,
    verbose=True
)

In [ ]:
# Extract key results for reference
run_level_acc = acc_results['run_level']
cond_level_acc = acc_results['condition_level']
cb_effects = acc_results['counterbalance']
stim_effects_acc = acc_results['stimulation_effects']
age_effects_acc = acc_results['age_effects']

# Quick summary
print("\nStimulation effect on accuracy:")
if 'accuracy' in stim_effects_acc:
    r = stim_effects_acc['accuracy']
    sig = '*' if r['p'] < 0.05 else ''
    print(f"  Sham: {r['sham_mean']:.3f}, Active: {r['active_mean']:.3f}, p = {r['p']:.3f}{sig}")

print("\nStimulation effect on win rate:")
if 'win_rate' in stim_effects_acc:
    r = stim_effects_acc['win_rate']
    sig = '*' if r['p'] < 0.05 else ''
    print(f"  Sham: {r['sham_mean']:.3f}, Active: {r['active_mean']:.3f}, p = {r['p']:.3f}{sig}")

---
## 15. Electric Field Analysis

Analyzes simulated electric field strength in the target ROI (left DLPFC / F3):
- **Age × E-field**: Does cortical atrophy reduce field strength in older adults?
- **E-field moderation**: Does stronger e-field predict larger tACS effects?

In [ ]:
# Import e-field analysis module
from efield_analysis import run_efield_analysis

In [ ]:
# Run e-field analysis
# Requires: efield_roi_summary.csv in working directory

efield_results = run_efield_analysis(
    efield_path='efield_roi_summary.csv',
    cond_df=acc_results['condition_level'],
    wsls_h2=wsls_results.get('wsls_h2'),
    rw_df=rw_results.get('rw_mle'),
    ddm_df=ddm_results.get('ddm_subject') if ddm_results else None,
    show_plots=True,
    verbose=True
)

In [ ]:
# Extract key results
efield_df = efield_results['efield_data']
age_efield = efield_results['age_relationship']
efield_moderation = efield_results['moderation']

# Summary
print("\nE-field Analysis Summary:")
print(f"  Subjects with e-field data: {len(efield_df)}")

if age_efield:
    sig = '*' if age_efield['p'] < 0.05 else ''
    print(f"  Age × E-field: r = {age_efield['r']:.3f}, p = {age_efield['p']:.3f}{sig}")

print("\n  E-field moderation of Δ(active - sham):")
for dv, res in efield_moderation.items():
    sig = '*' if res['p'] < 0.05 else ''
    print(f"    {dv}: r = {res['r']:.3f}, p = {res['p']:.3f}{sig}")

---
## 15b. Update Master CSV with Post-Merge Variables

Adds variables from sections that run **after** the initial cognitive merge:
- Extended R-W parameters (α⁺, α⁻, τ, confirmation bias index, best model)
- Accuracy/win-rate condition-level summaries
- E-field ROI strength

In [ ]:
# -------------------------------------------------------------------------
# Update master CSV with variables computed AFTER the initial cognitive merge
# -------------------------------------------------------------------------
# The cognitive merge (Section 9) writes the master CSV, but several analyses
# run after it and produce additional subject-level variables. This cell
# appends those to subj_df and re-exports the CSV.

update_cols_added = []

# --- Extended R-W parameters ---
if REFIT_EXTENDED and extended_model_fits is not None and len(extended_model_fits) > 0:
    # extended_model_fits can be:
    #   (a) a dict keyed by model name → DataFrame of per-subject/condition fits
    #   (b) a DataFrame with subject_id, condition, and param columns
    # We handle both cases.

    # Step 1: Get the best-model fits as a single DataFrame
    ext_fits_df = None

    if isinstance(extended_model_fits, dict):
        # Dict of model_name → DataFrame
        # Use the best model per subject from extended_comparison, or fall back
        # to the most complex model (asymmetric_sticky > asymmetric > standard_sticky)
        model_priority = ['asymmetric_sticky', 'asymmetric_rw_sticky',
                          'asymmetric', 'asymmetric_rw',
                          'standard_sticky', 'rw_sticky']

        # Try to use best-model-per-subject if comparison table exists
        if (len(extended_comparison) > 0
                and 'best_model' in extended_comparison.columns
                and 'subject_id' in extended_comparison.columns):
            rows = []
            for _, row in extended_comparison.iterrows():
                bm = row['best_model']
                sid = str(row['subject_id'])
                if bm in extended_model_fits:
                    fits = extended_model_fits[bm]
                    if isinstance(fits, pd.DataFrame) and 'subject_id' in fits.columns:
                        subj_fits = fits[fits['subject_id'].astype(str) == sid]
                        for _, fr in subj_fits.iterrows():
                            rows.append(fr.to_dict())
            if rows:
                ext_fits_df = pd.DataFrame(rows)
        
        # Fallback: pick the first available model from priority list
        if ext_fits_df is None or len(ext_fits_df) == 0:
            for model_name in model_priority:
                if model_name in extended_model_fits:
                    candidate = extended_model_fits[model_name]
                    if isinstance(candidate, pd.DataFrame) and len(candidate) > 0:
                        ext_fits_df = candidate.copy()
                        print(f'  Using model: {model_name}')
                        break

            # Last resort: just grab the first dict value that's a DataFrame
            if ext_fits_df is None or len(ext_fits_df) == 0:
                for key, val in extended_model_fits.items():
                    if isinstance(val, pd.DataFrame) and len(val) > 0:
                        ext_fits_df = val.copy()
                        print(f'  Using model: {key} (first available)')
                        break

    elif isinstance(extended_model_fits, pd.DataFrame):
        ext_fits_df = extended_model_fits.copy()

    # Step 2: Extract per-condition parameters from ext_fits_df
    if ext_fits_df is not None and len(ext_fits_df) > 0:
        ext_fits_df['subject_id'] = ext_fits_df['subject_id'].astype(str)

        # Identify parameter columns (everything except metadata)
        meta_cols = {'subject_id', 'condition', 'model', 'model_name',
                     'nll', 'aic', 'bic', 'n_trials', 'converged'}
        param_cols = [c for c in ext_fits_df.columns if c not in meta_cols]

        # Normalize column names
        rename_map = {
            'alpha_plus': 'alpha_pos', 'alpha_minus': 'alpha_neg',
            'a_pos': 'alpha_pos', 'a_neg': 'alpha_neg',
            'alpha_p': 'alpha_pos', 'alpha_n': 'alpha_neg',
            'stickiness': 'tau', 'stick': 'tau', 'phi': 'tau',
        }
        ext_fits_df = ext_fits_df.rename(columns={
            k: v for k, v in rename_map.items() if k in ext_fits_df.columns
        })
        param_cols = [rename_map.get(c, c) for c in param_cols]
        param_cols = list(dict.fromkeys(param_cols))  # deduplicate

        has_condition = 'condition' in ext_fits_df.columns

        for cond in (['sham', 'active'] if has_condition else ['overall']):
            if has_condition:
                cond_fits = ext_fits_df[ext_fits_df['condition'] == cond].copy()
                prefix = f'{cond}_'
            else:
                cond_fits = ext_fits_df.copy()
                prefix = ''

            if len(cond_fits) == 0:
                continue

            for param in param_cols:
                if param in cond_fits.columns:
                    new_col = f'{prefix}ext_{param}'
                    if new_col not in subj_df.columns:
                        merge_df = cond_fits[['subject_id', param]].rename(
                            columns={param: new_col}
                        )
                        merge_df = merge_df.drop_duplicates(subset='subject_id')
                        subj_df = subj_df.merge(merge_df, on='subject_id', how='left')
                        update_cols_added.append(new_col)

            # Confirmation bias ratio
            pos_col = f'{prefix}ext_alpha_pos'
            neg_col = f'{prefix}ext_alpha_neg'
            if pos_col in subj_df.columns and neg_col in subj_df.columns:
                cb_col = f'{prefix}confirmation_bias'
                if cb_col not in subj_df.columns:
                    subj_df[cb_col] = (
                        subj_df[pos_col] / subj_df[neg_col].replace(0, np.nan)
                    )
                    update_cols_added.append(cb_col)

        n_ext = len([c for c in update_cols_added if 'ext_' in c or 'confirmation' in c])
        print(f'Extended R-W: added {n_ext} columns')

    # Best-fitting model label
    if len(extended_comparison) > 0 and 'best_model' in extended_comparison.columns:
        best_model_df = extended_comparison[['subject_id', 'best_model']].copy()
        best_model_df['subject_id'] = best_model_df['subject_id'].astype(str)
        best_model_df = best_model_df.drop_duplicates(subset='subject_id')
        if 'best_model' not in subj_df.columns:
            subj_df = subj_df.merge(best_model_df, on='subject_id', how='left')
            update_cols_added.append('best_model')

# --- Condition-level accuracy & win rate ---
if acc_results is not None and acc_results.get('condition_level') is not None:
    cond_acc = acc_results['condition_level']

    for cond in ['sham', 'active']:
        cond_data = cond_acc[cond_acc['condition'] == cond][['subject_id', 'accuracy', 'win_rate']].copy()
        cond_data = cond_data.rename(columns={
            'accuracy': f'{cond}_accuracy',
            'win_rate': f'{cond}_win_rate'
        })
        cond_data['subject_id'] = cond_data['subject_id'].astype(str)
        cond_data = cond_data.drop_duplicates(subset='subject_id')

        for col in [f'{cond}_accuracy', f'{cond}_win_rate']:
            if col not in subj_df.columns:
                update_cols_added.append(col)

        subj_df = subj_df.merge(cond_data, on='subject_id', how='left')

    # Delta accuracy
    if 'sham_accuracy' in subj_df.columns and 'active_accuracy' in subj_df.columns:
        if 'delta_accuracy' not in subj_df.columns:
            subj_df['delta_accuracy'] = subj_df['active_accuracy'] - subj_df['sham_accuracy']
            subj_df['delta_win_rate'] = subj_df['active_win_rate'] - subj_df['sham_win_rate']
            update_cols_added.extend(['delta_accuracy', 'delta_win_rate'])

    n_acc = len([c for c in update_cols_added if 'accuracy' in c or 'win_rate' in c])
    print(f'Accuracy: added {n_acc} columns')

# --- E-field ROI data ---
try:
    if efield_results is not None and efield_results.get('efield_data') is not None:
        ef_df = efield_results['efield_data'].copy()
        ef_df['subject_id'] = ef_df['subject_id'].astype(str)

        # Grab any efield-prefixed columns
        ef_cols = [c for c in ef_df.columns if c != 'subject_id' and c.startswith('efield')]

        # Also grab common e-field column names without prefix
        for alt in ['mean_field', 'max_field', 'median_field', 'roi_mean', 'roi_max']:
            if alt in ef_df.columns and alt not in ef_cols:
                ef_cols.append(alt)

        if ef_cols:
            ef_merge = ef_df[['subject_id'] + ef_cols].drop_duplicates(subset='subject_id')

            for col in ef_cols:
                if col not in subj_df.columns:
                    update_cols_added.append(col)

            subj_df = subj_df.merge(ef_merge, on='subject_id', how='left')
            print(f'E-field: added {len(ef_cols)} columns')
except NameError:
    print('E-field analysis not run — skipping')

# --- Re-export master CSV ---
if update_cols_added:
    from cognitive_merge import export_master_csv
    csv_path = export_master_csv(subj_df, verbose=True)
    print(f'\nUpdated master CSV with {len(update_cols_added)} new columns:')
    for col in update_cols_added:
        n = subj_df[col].notna().sum() if col in subj_df.columns else 0
        print(f'  {col}: {n}/{len(subj_df)}')
else:
    print('No new columns to add — master CSV unchanged')

In [ ]:
# Quick diagnostic: verify master CSV contents
if MASTER_CSV.exists():
    master_check = pd.read_csv(MASTER_CSV)
    print(f'Master CSV: {len(master_check)} subjects × {len(master_check.columns)} variables')
    print(f'\nVariable coverage (non-missing):')
    for col in master_check.columns:
        if col != 'subject_id':
            n = master_check[col].notna().sum()
            if n > 0:
                print(f'  {col}: {n}/{len(master_check)}')
else:
    print('Master CSV not yet created — run cognitive merge section first')

---
## Module Reference

| Module | Purpose |
|--------|--------|
| `config.py` | Paths, SUBJECT_INFO, colors, constants |
| `data_loading.py` | Load and preprocess behavioral data |
| `exclusions.py` | Apply behavioral and stim exclusion criteria |
| `sample_descriptives.py` | Demographics, age distribution, cap size |
| `blinding_analysis.py` | Signal detection analysis of blinding |
| `wsls.py` | Win-stay/lose-shift analysis (condition-level and per-run) |
| `reversal_analysis.py` | Reversal-locked accuracy curves |
| `rescorla_wagner.py` | R-W model fitting (MLE/MAP) |
| `rescorla_wagner_extended.py` | Extended R-W: asymmetric learning, stickiness, model comparison |
| `ddm.py` | Drift diffusion model fitting |
| `eeg_theta.py` | Theta reactivity from baseline EEG |
| `cognitive_merge.py` | External data integration, composites, master CSV export, missing data audit |
| `hypothesis_tests.py` | H1.1, H1.2, H2.1, H2.2, theta moderation |
| `order_effects.py` | Counterbalance and session effects |
| `correlations.py` | Correlation heatmap |
| `accuracy_analysis.py` | Run-level accuracy, learning curves, counterbalance, baseline moderation |
| `efield_analysis.py` | E-field strength, age effects, stimulation moderation |
| `plotting_utils.py` | Shared visualization utilities |

### Analysis Toggles (Cell 2)

| Toggle | Default | Effect |
|--------|---------|--------|
| `REFIT_RW` | `True` | `False` → load R-W params from master CSV |
| `REFIT_DDM` | `True` | `False` → load DDM params from master CSV |
| `REFIT_EXTENDED` | `True` | `False` → skip extended R-W models entirely |
| `INCLUDE_EXPLORATORY` | `False` | `True` → load TEI, AQ, PANAS, IOS, etc. |

### Master CSV Contents

The master CSV (`master_subject_data.csv`) is written in two stages:
1. **Section 9** (cognitive merge): Demographics, cognitive composites, survey measures, WSLS, R-W, DDM, theta
2. **Section 15b** (post-merge update): Extended R-W parameters, condition-level accuracy/win-rate, e-field ROI strength

---
## 16. Survey Measure Exploration

 Systematic exploration of all available survey and clinical measures
 against behavioral and computational DVs. This section:

 1. Computes pairwise correlations between every survey measure and every DV
 2. Applies FDR correction (Benjamini-Hochberg) separately for baseline and change DVs
 3. Generates heatmaps and scatter plots for the strongest relationships
 4. Runs regression follow-ups controlling for age and global cognition
 5. Tests whether any survey measures moderate the tACS effect

 **All analyses in this section are exploratory and should be interpreted accordingly.**

In [ ]:
from survey_exploration import run_survey_exploration
 
survey_results = run_survey_exploration(
    subj_df,
    show_plots=True,
    verbose=True,
    min_n=10,       # minimum n for a correlation to be computed
    fdr_alpha=0.05,
    max_scatter_plots=8
)
 
# Access results programmatically if needed:
# survey_results['corr_df']              — full correlation table
# survey_results['regression_results']   — follow-up regressions
# survey_results['moderation_hits']      — tACS moderation results
# survey_results['figures']              — dict of Plotly figures

In [ ]:
from loseshift_accuracy_diagnostic import diagnose_young_loseshift_accuracy
diagnose_young_loseshift_accuracy(subj_df)

In [ ]:
from baseline_moderation_diagnostic import run_baseline_moderation_diagnostic
results = run_baseline_moderation_diagnostic(subj_df, show_plots=True)

In [ ]:
# Merge DDM params into subj_df (bridge naming mismatch)
if ddm_results is not None and 'subject_params' in ddm_results:
    ddm_merge = ddm_results['subject_params'].copy()
    ddm_merge['subject_id'] = ddm_merge['subject_id'].astype(str)
    ddm_cols = [c for c in ddm_merge.columns if c != 'subject_id']
    # Drop any existing DDM columns to avoid conflicts
    subj_df = subj_df.drop(columns=[c for c in ddm_cols if c in subj_df.columns], errors='ignore')
    subj_df = subj_df.merge(ddm_merge, on='subject_id', how='left')
    print(f'Merged DDM params: {ddm_cols}')
    print(f'  Subjects with drift rate: {subj_df["ddm_v_sham"].notna().sum()}')

In [ ]:
## 16b. Age-Moderated Stimulation: Decomposing the Accuracy Finding

from age_moderated_stimulation import run_age_moderated_stimulation

age_stim_results = run_age_moderated_stimulation(
    data_clean=data_clean,
    subj_df=subj_df,
    h2_eligible=h2_eligible,
    wsls_h2=wsls_results.get('wsls_h2'),
    rw_mle=rw_results.get('rw_mle'),
    show_plots=True,
    verbose=True,
)

# Update subj_df with new columns (delta_rt, correct-stay decomposition)
subj_df = age_stim_results['subj_df']

In [ ]:
ddm_cols = [c for c in subj_df.columns if 'ddm' in c.lower() or 'drift' in c.lower() or '_v_' in c]
print(ddm_cols)

In [ ]:
## 16c. Baseline vs. Age: What Predicts Stimulation Response?

from baseline_vs_age import run_baseline_vs_age_analysis

bva_results = run_baseline_vs_age_analysis(
    data_clean=data_clean,
    subj_df=subj_df,
    h2_eligible=h2_eligible,
    show_plots=True,
    verbose=True,
)

subj_df = bva_results['subj_df']

---
## 17. Export Full Notebook Report

Exports all printed output, statistics, and plots to a single PDF report.
This document can be shared with collaborators or fed back for review.

In [ ]:
# =========================================================================
# Export notebook to PDF/HTML report
# =========================================================================
# This cell saves the fully-executed notebook as a self-contained report.
#
# Two export options:
#   1. HTML (recommended): Preserves Plotly interactivity, works everywhere
#   2. PDF: Static, requires nbconvert + chromium/wkhtmltopdf
#
# Usage: Run the full notebook top-to-bottom first, then execute this cell.
 
import subprocess
from pathlib import Path
from datetime import datetime
 
EXPORT_FORMAT = 'html'  # Change to 'pdf' if you have chromium installed
 
notebook_path = Path('main_analyses.ipynb')
output_dir = DATA_DIR.parent / 'reports'
output_dir.mkdir(exist_ok=True)
 
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
output_name = f'tacs_bandit_analysis_report_{timestamp}'
 
 
def fix_plotly_in_html(html_path):
    """
    Inject Plotly CDN config into the HTML so require(["plotly"]) resolves.
    
    nbconvert produces HTML that uses RequireJS to load Plotly, but never
    tells RequireJS where to find the Plotly module. This function inserts
    a require.config() block pointing to the Plotly CDN right after the
    require.js script tag, which makes all plots render correctly.
    """
    with open(html_path, 'r', encoding='utf-8') as f:
        html = f.read()
 
    plotly_config = """
<script type="text/javascript">
if (typeof require !== 'undefined') {
    require.config({
        paths: {
            plotly: 'https://cdn.plot.ly/plotly-2.35.2.min'
        }
    });
}
</script>
"""
 
    # Insert right after the require.js script tag
    target = 'require.min.js"></script>'
    if target in html:
        html = html.replace(target, target + plotly_config)
        with open(html_path, 'w', encoding='utf-8') as f:
            f.write(html)
        print(f'  ✓ Plotly CDN config injected into {Path(html_path).name}')
        return True
    else:
        print(f'  ⚠ Could not find require.js tag in {Path(html_path).name}')
        print(f'    Plots may not render. Check that nbconvert included require.js.')
        return False
 
 
if EXPORT_FORMAT == 'html':
    output_file_no_code = output_dir / f'{output_name}.html'
    output_file_with_code = output_dir / f'{output_name}_with_code.html'
    
    # Export without code (clean report for sharing)
    cmd_no_code = [
        'jupyter', 'nbconvert',
        '--to', 'html',
        '--no-input',
        '--output', str(output_file_no_code),
        str(notebook_path)
    ]
    
    # Export with code (for review / reproducibility)
    cmd_with_code = [
        'jupyter', 'nbconvert',
        '--to', 'html',
        '--output', str(output_file_with_code),
        str(notebook_path)
    ]
    
    try:
        # --- No-code version ---
        result = subprocess.run(cmd_no_code, capture_output=True, text=True)
        if result.returncode == 0:
            print(f'Report exported (no code): {output_file_no_code}')
            fix_plotly_in_html(output_file_no_code)
        else:
            print(f'Export failed (no code): {result.stderr}')
        
        # --- With-code version ---
        result2 = subprocess.run(cmd_with_code, capture_output=True, text=True)
        if result2.returncode == 0:
            print(f'Report exported (with code): {output_file_with_code}')
            fix_plotly_in_html(output_file_with_code)
        else:
            print(f'Export failed (with code): {result2.stderr}')
            
    except FileNotFoundError:
        print('nbconvert not found.')
        print('Install with: pip install nbconvert')
 
elif EXPORT_FORMAT == 'pdf':
    output_file = output_dir / f'{output_name}.pdf'
    cmd = [
        'jupyter', 'nbconvert',
        '--to', 'webpdf',
        '--no-input',
        '--output', str(output_file),
        str(notebook_path)
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            print(f'PDF report exported: {output_file}')
        else:
            print(f'PDF export failed: {result.stderr}')
            print('PDF export requires chromium. Try: playwright install chromium')
    except FileNotFoundError:
        print('nbconvert not found.')
        print('Install with: pip install nbconvert')
 
print(f'\nReport directory: {output_dir}')
print(f'Master CSV: {MASTER_CSV}')

In [ ]:
from presentation_plots import generate_all_figures

# For all 7 figures (need trial_df for learning curves):
generate_all_figures(subj_df, trial_df=bandit_data, output_dir='figures/')

# Or just the subject-level plots (skips learning curves):
generate_all_figures(subj_df, output_dir='figures/')

In [ ]:
generate_all_figures(subj_df, trial_df=data, output_dir='figures/')